# Pipeline Run

Thin Jupyter front-end to the `spot-detector` pipeline. Running every cell top to
bottom is equivalent to `uv run spot-detector configs/config.yml`, with the config,
run-summary figure, and output tables shown inline instead of only written to disk.

**Before running:**

- `uv sync` has been run, and this notebook is on that environment's kernel
- `configs/config.yml` points at the raw data and models you want
  (paths, `mode.do_3d`, channel indices, thresholds)

Outputs land in `output/` (`tables/`, `figures/`, `logs/`); files for the same mode
are overwritten, so re-running is safe.


## 1. Setup

Imports, then `chdir` to the project root. `load_config` validates
`paths.raw_data_dir` and `detection.spotiflow_model_path` as real directories
_relative to the current working directory_, so the notebook must run from the repo
root no matter where Jupyter was launched. The `project_root` walk up from
`Path.cwd()` makes that work whether the kernel starts in `notebooks/` or the root.


In [ ]:
import os
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from ipyfilechooser import FileChooser
from IPython.display import Image, display
from itables import init_notebook_mode

from spot_detector.config import load_config
from spot_detector.run_pipeline import run_pipeline

init_notebook_mode(all_interactive=True, connected=False)

In [ ]:
project_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/config.yml").exists()
)
os.chdir(project_root)

## 2. Load and Review the Config

`load_config` returns a validated, frozen `PipelineConfig`. The table below is the
full resolved config — scan it before running. A wrong `mode.do_3d`, channel index,
or model path is far cheaper to catch here than after a long run. To change
anything, edit `configs/config.yml` and re-run this section.


In [ ]:
config_path = Path("configs/config.yml")
config = load_config(config_path)

out_dir = Path(config.paths.out_dir)
mode = "3D" if config.mode.do_3d else "2D"

In [ ]:
cfg_df = pd.DataFrame(
    [
        {"section": section, "key": key, "value": value}
        for section, params in config.model_dump(mode="json").items()
        for key, value in params.items()
    ]
).set_index(["section", "key"])
cfg_df

## 3. Run the Pipeline

Segments objects (Cellpose-SAM), detects spots (Spotiflow), assigns spots to
objects, measures per-object morphology, and writes per-scene QC figures plus
per-condition and run-level CSVs. The progress bar steps per file; detailed logs
stream to `output/logs/run_<timestamp>.log`.

The logging cell is guarded (`isinstance(h, logging.FileHandler)` check) so
re-running it won't stack duplicate handlers or open a second log file —
`configure_logging` is not idempotent on its own.


In [ ]:
import logging

from spot_detector.cli import configure_logging

if not any(isinstance(h, logging.FileHandler) for h in logging.getLogger().handlers):
    configure_logging(Path(config.paths.out_dir) / "logs")

In [ ]:
run_df = run_pipeline(config=config)

## 4. Run Summary

One figure + table per run, aggregating every condition. Produced whenever the run
wrote at least one object; otherwise the guard message tells you to run section 3.


In [ ]:
tab_path = out_dir / "tables" / f"_run_objects_{mode}.csv"
fig_path = out_dir / "figures" / f"_run_summary_{mode}.png"
if tab_path.exists() and fig_path.exists():
    display(pd.read_csv(tab_path))
    display(Image(str(fig_path), width=1000))
else:
    print("Pipeline needs to be run first!")

## 5. Browse Individual Outputs

Pick any per-condition CSV or per-scene QC PNG from `output/` to render it inline —
CSV as an interactive table, PNG as an image. Complements section 4, which always
shows the run-level rollup.


In [ ]:
fc = FileChooser(str(out_dir), filter_pattern=["*.csv", "*.png"], select_default=False)
view = widgets.Output()


def on_pick(chooser):
    if chooser.selected is None:
        return
    p = Path(chooser.selected)
    view.clear_output(wait=True)
    with view:
        display(pd.read_csv(p)) if p.suffix == ".csv" else display(
            Image(str(p), width=1000)
        )


fc.register_callback(on_pick)
display(fc, view)